In [1]:
#! pip uninstall typing-extensions --yes
#! pip install typing-extensions==4.14.0 

Found existing installation: typing_extensions 4.7.1
Uninstalling typing_extensions-4.7.1:
  Successfully uninstalled typing_extensions-4.7.1
     ---------------------------------------- 43.8/43.8 kB 1.1 MB/s eta 0:00:00


In [33]:
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
from lxml import html

import requests
from bs4 import BeautifulSoup
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
#!pip install requests_html
#from requests_html import HTMLSession
import random
import re
import time as  t


import string
import matplotlib as mlt
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import LabelEncoder

import pymysql
pymysql.install_as_MySQLdb()
import MySQLdb

# Variable Declaration
global bullet_characters 
global salary_descriptors 
global requirement_descriptors
bullet_characters = ['-', '•', '◦', '▪', '▸', '➤', '★', '*']
salary_descriptors = ['$','pay','paid','salary','compensation']
requirement_descriptors = '(?i)requ|respon|qual|look|skill|know|job|function|Elig|edu|exper|duties|duty'
number = LabelEncoder()

In [2]:
def merge(dict1, dict2):
    return(dict2.update(dict1))

# Extract HTML - Paginated Func
def extract(result_start):    
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States&start={result_start}"
    
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"]
    
    headers = {"accept":"*/*",
               "accept-encoding": "gzip, deflate, br, zstd",
               "accept-language": "en-US,en;q=0.9",
               'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

def extract_inner(url_inner):
       
    user_agents_list = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"]
    
    
    headers = {"accept":"*/*",
               "accept-encoding": "gzip, deflate, br, zstd",
               "accept-language": "en-US,en;q=0.9",
               'User-Agent': random.choice(user_agents_list)}

    r_inner = requests.get(url_inner,headers)
    soup_inner = BeautifulSoup(r_inner.content,'html.parser')
    return(soup_inner)

def transform(soup):
    divs = soup.find_all('li')
    
    for job in divs:
        raw_job={         
                'title': job.find('h3',class_='base-search-card__title').get_text().strip() if job.find('h3',class_='base-search-card__title') else None,
                'company': job.find('h4',class_ = 'base-search-card__subtitle').get_text().strip() if job.find('h4',class_ = 'base-search-card__subtitle') else None,
                'location_info': job.find('span',class_ = 'job-search-card__location').get_text().strip() if job.find('span',class_ = 'job-search-card__location') else None,
                'posting_date': job.find('time').get_text().strip() if job.find('time') else None
                }
        raw_job['job_url'] = job.a['href'].partition('?position')[0] if job.a['href'] else None 
        raw_job['job_id'] = raw_job['job_url'].split('-')[-1]
    
        raw_job['scrape_datetime'] = datetime.now()         
        
        #details = []
        more_info = extract_inner(raw_job['job_url'])
        raw_job['details'] = detail_fetcher(more_info)
        
        jobs_list.append(raw_job)

    return


In [116]:
# For final intergration for chunk in range(0, 241,10):
jobs_list = []
i = 1
for chunk in range(0, 11,10):
    recent_jobs_html = extract(chunk)
    transform(recent_jobs_html)
    t.sleep(4)# adding this due to a limit on the scrape frequency
    print(f'Through {i} iteration(s) we have yielded {len(jobs_list)} results')
    i+=1

Through 1 iteration(s) we have yielded 10 results
Through 2 iteration(s) we have yielded 20 results


In [18]:
jobs_list = []
c = extract(250)

In [19]:
transform(c)

[<strong>Who We Are<br/><br/></strong>, <strong>Why Work for the USTA?<br/><br/></strong>, <strong>The Role<br/><br/></strong>, <strong>Responsibilities<br/><br/></strong>, <strong>Who You Are<br/><br/></strong>, <strong>What We Offer<br/><br/></strong>, <strong>$50,000 - $55,000.</strong>, <strong> Come One, Come All <br/><br/></strong>]
True
[<strong>What makes a TOCA Teammate? An individual that seeks to...<br/><br/></strong>, <strong>Job Description Highlights:<br/><br/></strong>, <strong>Reports To:</strong>, <strong>Location: </strong>, <strong>Flexible Schedule:</strong>, <strong>Role Scope and Requirements:<br/><br/></strong>, <strong>Knowledge and Experience: <br/><br/></strong>]
True
[<strong>JH ASSISTANT FOOTBALL COACH<br/><br/></strong>, <strong>LEVEL D<br/><br/></strong>, <strong>JONATHAN ALDER LOCAL SCHOOLS<br/><br/></strong>, <strong>TITLE: Assistant Athletic Coach - supplemental position<br/><br/></strong>, <strong>SUGGESTED<br/><br/></strong>, <strong>QUALIFICATIONS: 1

[<strong>About the Kings League</strong>, <strong>About you</strong>, <strong>Partnerships Director</strong>, <strong>strategic digital sponsorships, partnerships, and monetization of our media and IP assets</strong>, <strong>sports and digital sector</strong>, <strong>commercial impact</strong>, <strong>Main Responsibilities</strong>, <strong>digital and online opportunities</strong>, <strong>digital sponsorships, branded content, and online partnerships</strong>, <strong>digital advertisers, sports agencies, and online media partners</strong>, <strong>digital commercial offers</strong>, <strong>digital sports sponsorship, entertainment, and online media</strong>, <strong>Requirements</strong>, <strong>digital sponsorship sales, business development, or strategic partnerships</strong>, <strong>digital and online deals</strong>, <strong>sports brands, digital agencies, and media buyers</strong>, <strong>digital sports agency or managing digital media sales.</strong>, <strong><em>Divers

In [34]:
#### I AM NOW WORKING HERE TO BETTER ADDRESS SOME ASPECTS OF THE DETAIL FETCHING. I NEED TO ADJUST THE REGEX CATCH ALL TO PROPERLY PARSE AND WRAP THE PROSE EVEN IF NOT BULLETED (WHICH ITS NOT)
# Regex pattern covering headers that describe role requirements/responsibilities
PROSE_HEADER_PATTERN = re.compile(
    requirement_descriptors
)

def resolve_list_tag(tag):
    # Helper: find a ul/ol — checks inside tag first, then next sibling
    inner_list = tag.find('ul') or tag.find('ol')
    if inner_list:
        return(inner_list)
    sibling = tag.find_next_sibling()
    if sibling and sibling.name in ['ul', 'ol']:
        return(sibling)
    return(None)


def detail_fetcher(temp_text):
    try:
        job_inner = temp_text.find('div', class_="show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden")

        if not job_inner:
            print("detail_fetcher: target div not found")
            return([])

        job_details = job_inner.find_all('strong')

        if job_details:
            p_tag = bool(job_inner.find('ul'))
        else:
            p_tag = False

        if p_tag:
            pass
        else:
            job_details = wrap_bullets(job_inner)

        data = {}
        for section in job_details:
            if not section:
                continue

            header_text = section.get_text().strip()

            # strong tags are wrapped in <p> — walk up to parent first
            parent = section.parent
            if parent and parent.name == 'p':
                next_tag = parent.find_next_sibling()
            else:
                next_tag = section.find_next_sibling()

            if not next_tag:
                continue

            # Skip blank spacer tags e.g. <p><br/></p>
            while next_tag and next_tag.name == 'p' and not next_tag.get_text(strip=True):
                next_tag = next_tag.find_next_sibling()

            if not next_tag:
                continue

            # Structured list — always capture regardless of header
            if next_tag.name in ['ul', 'ol']:
                items = next_tag.find_all('li')
                data[header_text] = [item.get_text(strip=True) for item in items]

            # Prose <p> — check against regex pattern before capturing
            elif next_tag.name == 'p':
                list_tag = resolve_list_tag(next_tag)
                if list_tag:
                    # <p> is an intermediary before a list
                    items = list_tag.find_all('li')
                    data[header_text] = [item.get_text(strip=True) for item in items]
                elif PROSE_HEADER_PATTERN.search(header_text):
                    # [REGEX WHITELIST] Capture prose only for requirement/responsibility headers
                    prose_text = next_tag.get_text(strip=True)
                    data[header_text] = [prose_text]

        details_df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})

    except Exception as e:
        print(f"detail_fetcher error: {e}")
        details_df = []

    return(details_df)

In [8]:
print(temp_text)

<!DOCTYPE html>

<html lang="en">
<head>
<meta content="d_jobs_guest_details" name="pageKey"/>
<meta content="max-image-preview:large, noarchive" name="robots"/>
<meta content="max-image-preview:large, archive" name="bingbot"/>
<!-- --><!-- --> <meta content="en_US" name="locale"/>
<!-- --> <meta data-app-version="2.0.2910" data-browser-id="14da24a1-829a-43d2-862f-0a7b77d9e4bd" data-call-tree-id="AAZQhz5tr8Uo/VdJImc0Mg==" data-dfp-member-lix-treatment="control" data-disable-jsbeacon-pagekey-suffix="false" data-dna-member-lix-treatment="enabled" data-enable-page-view-heartbeat-tracking="" data-human-member-lix-treatment="enabled" data-is-bot="true" data-is-epd-audit-event-enabled="false" data-is-feed-sponsored-tracking-kill-switch-enabled="false" data-member-id="0" data-multiproduct-name="jobs-guest-frontend" data-network-interceptor-lix-value="control" data-page-instance="urn:li:page:d_jobs_guest_details;Dr9LMCkGS5S7KVBb3jdWMQ==" data-recaptcha-v3-integration-lix-value="control" data-s

In [35]:
temp_text = extract_inner('https://www.linkedin.com/jobs/view/na-sponsorship-coordinator-at-sage-4402926589/')
detail_fetcher(temp_text)

,What are we looking for?,What's in it for you?
0,"You’ll bring ideally 3 years experience in either in house or agency, working with major brands or rights holders, operating as a slick, highly organised project manager who can lead complex events from concept to delivery. With a strong passion for sports and entertainment, you pair creative thinking and compelling storytelling with the ability to make quick, insight driven decisions that move work forward. You’ll be confident guiding cross functional teams through the sponsorship journey, influencing stakeholders and ensuring flawless execution at every stage. Your ability to stay calm under pressure, manage multiple workstreams and maintain exceptional attention to detail will be key to delivering standout activations that elevate our brand and create memorable experiences.",Competitive annual bonuses
1,NaN,"Comprehensive health, dental, and vision coverage"
2,NaN,401(k) retirement match (100% matching up to 4%)
3,NaN,32 days paid time off (22 personal days & 10 national holidays)
4,NaN,18 weeks of paid parental leave (offered 1 year after the start date)
5,NaN,Work Away Program: Opportunity to work & play for 10 weeks from another country (Sage-approved list)
6,NaN,Sage Foundation: 5 days paid yearly to volunteer
7,NaN,"$5,250 tuition reimbursement per calendar year starting 6 months after the hire date"
8,NaN,Sage Wellness Rewards Program ($600 wellness credit and $360 fitness reimbursement annually)


In [24]:
linkedin_df = pd.DataFrame(jobs_list)
linkedin_df.head()

title  \
0  Senior Coordinator, Global Event Services & Athlete Experience   
1                                    Seasonal Baseball Camp Coach   
2                                               JH Football Coach   
3         Field Marketing Manager - Columbus & Cleveland (REMOTE)   
4                                           Tennis Tech Installer   

                                   company   location_info posting_date  \
0  (USTA) United States Tennis Association     Orlando, FL   3 days ago   
1                            TOCA Football    Marietta, GA   4 days ago   
2     Jonathan Alder Local School District  Plain City, OH    1 day ago   
3                    DICK'S Sporting Goods   United States   3 days ago   
4                                     NCAA     Chicago, IL  7 hours ago   

                                                                                                                                              job_url  \
0  https://www.linkedin.com/jobs/view/senior-coordinator-global-event-services-athlete-experience-at-usta-united-states-tennis-association-4393814295   
1                                                         https://www.linkedin.com/jobs/view/seasonal-baseball-camp-coach-at-toca-football-4407578144   
2                                             https://www.linkedin.com/jobs/view/jh-football-coach-at-jonathan-alder-local-school-district-4405176814   
3                            https://www.linkedin.com/jobs/view/field-marketing-manager-columbus-cleveland-remote-at-dick-s-sporting-goods-4397837022   
4                                                                         https://www.linkedin.com/jobs/view/tennis-tech-installer-at-ncaa-4406298219   

       job_id            scrape_datetime  \
0  4393814295 2026-04-28 12:24:44.172268   
1  4407578144 2026-04-28 12:24:45.916034   
2  4405176814 2026-04-28 12:24:46.565098   
3  4397837022 2026-04-28 12:24:47.455539   
4  4406298219 2026-04-28 12:24:48.250536   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [29]:
# Data Cleaning Section - I believe all of this functionally works so far. I need to address hybrid/remote location tags and see if I want to implement them. 
linkedin_df = pd.DataFrame(jobs_list)

linkedin_df['city'] = linkedin_df['location_info'].str.split(',').str[0].str.strip()
linkedin_df['state'] = linkedin_df['location_info'].str.split(',').str[1].str.strip()
linkedin_df['country'] = 'United States'
linkedin_df.loc[linkedin_df['city'].str.contains("[,]"),'city'] = linkedin_df['city'].str.split(',').str[0].str.strip()
linkedin_df.loc[linkedin_df['city'].str.len()==2,'state'] = linkedin_df.city
linkedin_df.loc[linkedin_df['state'].str.contains('[ ]'),'country'] = linkedin_df.state
linkedin_df.loc[linkedin_df['state'].str.contains('[ ]'),'state'] = None 
linkedin_df['remote_option'] = linkedin_df['location_info'].str.contains(' Rem| Hyb', case=False, na=False)

#linkedin_df["company_id"] = number.fit_transform(linkedin_df["company"].astype('str'))
#linkedin_df.loc[linkedin_df['company_id'] == 0,'company_id'] = (max(linkedin_df['company_id'])+1)
#linkedin_df.sort_values(by = 'company_id')

#linkedin_df.loc[~linkedin_df['posting_date_identifier'].str.endswith('s'),'posting_date_identifier'] = linkedin_df['posting_date_identifier'] + ('s')
linkedin_df['posting_source_ID'] = 2
linkedin_df
#linkedin_df['posting_timestamp'] = linkedin_df.apply(
#    lambda row: row['scrape_datetime'] - timedelta(**{row['posting_date_identifier']: int(row['posting_numbers'])}),
#    axis=1)
#linkedin_df.loc[linkedin_df['remote_option'] == True]
#linkedin_df[linkedin_df.duplicated(subset=['job_id'])]
#linkedin_df.loc[linkedin_df['title'].str.contains("\(")]
#linkedin_df[linkedin_df.job_id == '2165207']

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [ ]:
joblist = []
leagues = ['Major%20League%20Soccer', 'Major%20League%20Baseball','National%20Football%20League']
for league in leagues:
    c=extract(league)
    transform(c)
    
joblist1 = joblist

In [ ]:
joblist = []
leagues_again = ['National%20Hockey%20League', 'National%20Basketball%20Association']
for league in leagues_again:
    c=extract(league)
    transform(c)
    
joblist2 = joblist

In [ ]:
database = MySQLdb.connect(host="localhost" , user="root" , passwd="Pps11844")
cursor = database.cursor()

def execute_query(query_statement):
    try:
        cursor.execute(query_statement);
        database.commit();
        print("Data is Succefully Inserted")
    
    except Exception as e :
        database.rollback();
        print("The  Exception Occured : ", e)

execute_query("USE JobsinSports")

SQL_df_posting = pd.read_sql('select * from job_posting',database)

SQL_df_companies = pd.read_sql('select * from company_team',database)

cursor.execute("SELECT MAX(company_ID) FROM company_team;")
result = cursor.fetchone();
max_comp_ID = result[0]

database.close()

In [ ]:
job_posting_linkedin_1 = pd.DataFrame(joblist1)
job_posting_linkedin_2 = pd.DataFrame(joblist2)
job_posting_linkedin = pd.concat([job_posting_linkedin_1, job_posting_linkedin_2])
job_posting_linkedin.reset_index(drop=True, inplace=True)

job_posting_linkedin['job_ID'] = job_posting_linkedin['job_ID'].astype(float)

for i,j in job_posting_linkedin.iterrows():
    if(re.findall(r'\$',j['additionals'])):
        job_posting_linkedin.at[i,'salary'] = '$'+(j['additionals'].partition('$')[2].partition('.')[0].partition('<')[0].partition('/')[0].partition('\\')[0].replace(' ','').replace('to','-').replace('TO','-').replace('K',',000').replace('k',',000').replace('\n',',000'))
    else:
        job_posting_linkedin.at[i,'salary'] = 'NA'

for i,j in job_posting_linkedin.iterrows():
    if(len(j['salary'])==3):
        job_posting_linkedin.at[i,'salary'] = (j['salary'] + '/hr')
    else:
        pass

for i,j in job_posting_linkedin.iterrows():
    if(re.findall(r'New York City',j['Location'])):
        job_posting_linkedin.at[i,'Location'] = 'New York, NY'
    else:
        pass
    
job_posting_linkedin['job_city'] = job_posting_linkedin['Location'].str.partition(",")[0]
job_posting_linkedin['job_state'] = job_posting_linkedin['Location'].str.partition(",")[2]

job_posting_linkedin['Company'] = job_posting_linkedin['Company'].str.partition('(')[0].str.replace('Football Club ','FC').str.replace('Football Club','FC').str.replace('Soccer Club','SC')
job_posting_linkedin['Company'] = job_posting_linkedin['Company'].str.strip()
job_posting_linkedin['posting_source_ID'] = 3
job_posting_linkedin['application_deadline'] = 'Unknown'
job_posting_linkedin['scrape_datetime'] = pd.to_datetime(job_posting_linkedin['scrape_datetime'])
job_posting_linkedin['posting_datetime'] = pd.to_datetime(job_posting_linkedin['posting_datetime'])


job_requirements_df = pd.DataFrame(job_posting_linkedin[['job_ID','details']])
job_requirements_df_final = job_requirements_df.assign(temp = job_requirements_df.details.str.split(",")).explode('details').drop('temp',axis=1)
job_requirements_df_final['details'] = job_requirements_df_final['details'].str.replace("'","").str.replace('"','')

In [ ]:
job_posting_linkedin.drop(['Location','details','additionals'],axis = 1,inplace = True)

In [ ]:
Company_Team = pd.DataFrame(job_posting_linkedin['Company'])
Company_Team_df = Company_Team.drop_duplicates()

In [ ]:
Company_Team_df['Company_temp'] = [1,2,3,4,5,6,7,8]
Company_Team_df.loc[Company_Team_df['Company_temp'] == 1,'company_ID'] = int(max_comp_ID + 1)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 5,'company_ID'] = int(max_comp_ID + 2)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 6,'company_ID'] = int(max_comp_ID + 3)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 7,'company_ID'] = int(max_comp_ID + 4)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 8,'company_ID'] = int(max_comp_ID + 5)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 2,'company_ID'] = 258
Company_Team_df.loc[Company_Team_df['Company_temp'] == 3,'company_ID'] = 257
Company_Team_df.loc[Company_Team_df['Company_temp'] == 4,'company_ID'] = 255
Company_Team_df.drop('Company_temp',inplace=True,axis=1)

In [ ]:
Company_Team_df

In [ ]:
job_posting_linkedin_df = pd.merge(job_posting_linkedin, Company_Team_df, left_on="Company", right_on="Company", how='left')

In [ ]:
job_posting_linkedin_df = job_posting_linkedin_df.reindex(columns = ['job_ID','title',"company_ID",'posting_source_ID','posting_datetime','scrape_datetime','application_deadline', 'salary','job_city','job_state','url'])

In [ ]:
Sources = pd.DataFrame({'source_ID': [3], 'source_name': ['Linkedin']})

In [ ]:
# Tested but not perfected yet -- IGNORE
#job_posting_linkedin_df
#count = 1

#for i,j in Company_Team_df.iterrows():
#  print(i, j['Company'])
#    if (j['Company'] in SQL_df_companies['company_name'].values):
#        j.at[i,'company_ID'] = SQL_df_companies['company_ID']
#    else:
#        Company_Team_df.at[i,'company_ID'] = max_comp_ID + count
#        count = count + 1


In [ ]:
## Initialize connection to MYSQL
database = MySQLdb.connect(host="localhost" , user="root" , passwd="Pps11844")
cursor = database.cursor()

In [ ]:
def execute_query(query_statement):
    try:
        cursor.execute(query_statement);
        database.commit();
        print("Data is Succefully Inserted")
    
    except Exception as e :
        database.rollback();
        print("The  Exception Occured : ", e)


In [ ]:
execute_query("USE JobsinSports")


In [ ]:
for i,j in job_requirements_df_final.iterrows():
    execute_query('INSERT INTO Job_Requirements (job_ID, requirements) VALUES (%d, "%s")' % (j['job_ID'],j['details']))

In [ ]:
for i,j in Sources.iterrows():
    execute_query('INSERT INTO Sources (source_ID, source_name) VALUES (%d, "%s")' % (j['source_ID'],j['source_name']))

In [ ]:
for i,j in Company_Team_df.iterrows():
    execute_query('INSERT INTO Company_Team (company_ID, company_name) VALUES (%d, "%s")' % (j['company_ID'], j['Company']))

In [ ]:
for i,j in job_posting_linkedin_df.iterrows():
    execute_query('INSERT INTO Job_Posting (job_ID, job_title, company_ID, posting_datetime, scraped_datetime, salary, job_city, job_state, posting_url, source_identifier) VALUES (%d, "%s", %d, "%s","%s", "%s", "%s", "%s", "%s", %d)' % (j['job_ID'], j['title'], j['company_ID'], j['posting_datetime'], j['scrape_datetime'], j['salary'], j['job_city'], j['job_state'], j['url'], j['posting_source_ID']))

In [ ]:
database.close()

THIS IS UNUSED BUT INTERESTING CODE ID LIKE TO RETAIN.

I ran into issues becacuse instead of having paginated data the data was on an infinite scroll. To address this I 
implemented Selenium but ran into issues with the scrape as a user auth was needed. I went around this with an automated
escape action but the issue persisted as I wasnt getting any scroll. I was able to accheive a scroll by increasing scroll size 
and wait time to allow buffer.

The problem was I would get caught in an infinte buffer which yielded fewer results than I hoped for. I was able to find the succesful request which brought me to the Linkedin API which i used instead with the code herein. 

In [13]:
def extract():
    url = 'https://www.linkedin.com/jobs/search/?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States '
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    driver.get(url)
    
    actions = ActionChains(driver)
    actions.send_keys(Keys.ESCAPE).perform()

    last_height = 0
    unchanged_count = 0

    while True:
        # Scroll to the bottom
        driver.execute_script("window.scrollBy(0,5000)")
        time.sleep(5)  # Wait for new jobs to load

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(str(new_height)+"-"+str(last_height))

        if new_height == last_height:
            unchanged_count += 1
            print(f"No new content ({unchanged_count}/3)")
            if unchanged_count >= 3:
                print("Reached end of page.")
                break
        else:
            unchanged_count = 0
            last_height = new_height

    # Once fully scrolled, hand the full HTML to BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup

In [19]:
def extract():
    url = 'https://www.linkedin.com/jobs/search/?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States'
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    driver.get(url)
    time.sleep(5)  # Wait for initial page load
    
    actions = ActionChains(driver)
    actions.send_keys(Keys.ESCAPE).perform()
    
    last_height = driver.execute_script("return document.body.scrollHeight")  # Get initial height
    unchanged_count = 0
    current_position = 0

    while True:
        # --- Scroll down gradually in increments ---
        while current_position < last_height:
            current_position += 800  # Scroll 600px at a time
            driver.execute_script(f"window.scrollTo(0, {current_position});")
            time.sleep(0.8)  # Small pause between each step to let content trigger

        # --- Now at the bottom, wait for buffer to finish loading ---
        print("Waiting for buffer...")
        time.sleep(5)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(str(new_height) + "-" + str(last_height))

        if new_height == last_height:
            unchanged_count += 1
            print(f"No new content ({unchanged_count}/3)")
            if unchanged_count >= 3:
                print("Reached end of page.")
                break
        else:
            unchanged_count = 0
            last_height = new_height

    # Once fully scrolled, hand the full HTML to BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup

In [14]:
#### NEED TO REFINE THIS FUNCTION AGAIN. Right not I have an edge case where there is a strong tag in a p tag and the ul is after the p tag.

def detail_fetcher(temp_text):
    try:
        job_inner = temp_text.find('div', class_ = "show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden")
        #active_job_tag = (False if job_soup.find('div', class_ = "opportunity-preview__closed") else True)
        job_details = job_inner.find_all('strong')
        print(job_details)
        
        if job_details:
            if job_inner.find('ul'):
                p_tag = True
            else:
                p_tag = False
        else:
            p_tag = False
        
        print(p_tag)
              
        if (p_tag):
            pass
        else:
            job_details = wrap_bullets(job_inner)
        
        data = {}
        headers = []
        for section in job_details:
            if not section:
                continue
            
            details = []
                
            header_text = section.get_text()
            next_tag = section.find_next_sibling()
            
        
            if (next_tag and next_tag.name in ['ul', 'ol']):
                headers.append(header_text)
                items = next_tag.find_all('li')
                details = [item.get_text(strip=True) for item in items]
            
                data[header_text] = details 
                    
        
        # Convert to single-row DataFrame
        details_df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
    except:
        details_df = []

    return(details_df)

In [13]:
def wrap_bullets(html_text):
    if(html_text.find('ul')):
        for ul in html_text.find_all('ul'):
            for li in ul:
                li.string = '- ' + li.get_text()
    else:
        pass
    
    text = html_text.get_text(separator = '\n')
    
    lines = text.split('\n')
    result = []
    in_list = False
    
    for line in lines:
        stripped_line = line.strip()
        
        if in_list and stripped_line == '':
            continue
    
        if stripped_line and stripped_line[0] in bullet_characters:
            if not in_list:
                preceding_line = next((line for line in reversed(result) if line.strip()), None)
                result.append(f'<p>{preceding_line}</p>')
                result.append('<ul>')
                in_list = True
            content = stripped_line[1:].strip()
            result.append(f'<li>{content}</li>')
        else:
            if in_list:
                result.append('</ul>')
                in_list = False
            result.append(stripped_line)
            
    if in_list:
        result.append('</ul>')
    
    final_result = '\n'.join(result)
    
    soup = BeautifulSoup(final_result,'html.parser')

    return(soup.find_all('p'))